In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.3 MB/s eta 0:00:00


In [ ]:
import json
import os
import shutil
import random
from pathlib import Path

victim_dir = "/content/drive/MyDrive/Disaster_detection/victim_frames"

novictim_dir = "/content/drive/MyDrive/Disaster_detection/novictim_frames"

output_dataset = "/content/drive/MyDrive/Disaster_detection/victim_dataset"

folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for folder in folders:
    os.makedirs(os.path.join(output_dataset, folder), exist_ok=True)

victim_images = []
novictim_images = []

for ext in ["*.jpg", "*.jpeg", "*.png"]:
    victim_images.extend(Path(victim_dir).glob(ext))

for ext in ["*.jpg", "*.jpeg", "*.png"]:
    novictim_images.extend(Path(novictim_dir).glob(ext))

all_images = []

for img in victim_images:
    all_images.append(("victim", img))

for img in novictim_images:
    all_images.append(("novictim", img))

random.shuffle(all_images)

split_index = int(len(all_images) * 0.8)

train_images = all_images[:split_index]
val_images = all_images[split_index:]

def process_image(img_type, image_path, split):

    image_name = image_path.name

    dst_image = os.path.join(
        output_dataset,
        "images",
        split,
        image_name
    )

    shutil.copy(image_path, dst_image)

    txt_name = image_path.stem + ".txt"

    txt_output = os.path.join(
        output_dataset,
        "labels",
        split,
        txt_name
    )

    if img_type == "victim":

        json_name = image_path.stem + ".json"

        json_path = os.path.join(victim_dir, json_name)

        if os.path.exists(json_path):

            with open(json_path, "r") as f:
                data = json.load(f)

            image_width = data["imageWidth"]
            image_height = data["imageHeight"]

            with open(txt_output, "w") as txt_file:

                for shape in data["shapes"]:

                    if shape["shape_type"] != "polygon":
                        continue

                    class_id = 0

                    polygon = shape["points"]

                    normalized_points = []

                    for x, y in polygon:

                        x = x / image_width
                        y = y / image_height

                        normalized_points.append(str(x))
                        normalized_points.append(str(y))

                    line = str(class_id) + " " + " ".join(normalized_points)

                    txt_file.write(line + "\n")

    else:

        open(txt_output, "w").close()

for img_type, img_path in train_images:
    process_image(img_type, img_path, "train")

for img_type, img_path in val_images:
    process_image(img_type, img_path, "val")

print("Dataset preparation completed!")

Dataset preparation completed!


In [ ]:
data_yaml = """
path: /content/drive/MyDrive/Disaster_detection/victim_dataset

train: images/train
val: images/val

names:
  0: victim
"""

yaml_path = "/content/drive/MyDrive/Disaster_detection/data.yaml"

with open(yaml_path, "w") as f:
    f.write(data_yaml)

print("data.yaml created successfully!")
print(yaml_path)

data.yaml created successfully!
/content/drive/MyDrive/Disaster_detection/data.yaml


In [ ]:
from ultralytics import YOLO
import os

save_project = "/content/drive/MyDrive/Disaster_detection/training_results"

os.makedirs(save_project, exist_ok=True)

model = YOLO("yolo11s-seg.pt")

results = model.train(
    data="/content/drive/MyDrive/Disaster_detection/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    workers=2,

    project=save_project,
    name="victim_segmentation",

    exist_ok=True,
    save=True
)

print("Training completed!")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Disaster_detection/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=victim_segmentation, nbs=64, nms=False, opset=None, optimize=False, optimizer

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/Disaster_detection/training_results/victim_segmentation/weights/best.pt"
)

results = model.predict(
    source="/content/drive/MyDrive/Disaster_detection/test.mp4",
    save=True,
    imgsz=320,
    vid_stride=10,
    device=0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/4526) /content/drive/MyDrive/Disaster_detection/test.mp4: 192x320 (no detections), 86.4ms
video 1/1 (frame 2/4526) /content/driv

In [ ]:
!ls -lh /content/runs/segment/predict

total 104M
-rw-r--r-- 1 root root 104M May 30 06:08 test.avi


In [12]:
from IPython.display import Video

Video(
    "/content/runs/segment/predict/test.avi",
    embed=True
)

Buffered data was truncated after reaching the output size limit.